In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dask.distributed import Client, LocalCluster

client = Client(  # Note that `memory_limit` is the limit **per worker**.
    n_workers=4, threads_per_worker=1, memory_limit="2GB"
)
client  # If you click the dashboard link in the output, you can monitor real-time progress and get other cool visualizations.

In [ ]:
import copy
import sys
import xarray as xr
import numpy as np
import dask.array as da

import matplotlib.pyplot as plt
import hvplot.xarray
import scipy.constants

sys.path.append("..")
import processing_dask as pr
import multi_channel_processing as mpr
import plot_dask
import processing as old_processing

sys.path.append("../../preprocessing/")
from generate_chirp import generate_chirp

In [ ]:
plt.rcParams.update({"font.size": 12})  # set fontsize

### Open and resave file

In [ ]:
# file path to data and configs
PREFIX = "../../data/inFrontEBrain/20251217_143153"

# if multi channel data is in a single file interleaved together (the standard save format for ORCA) it needs to be split and resaved
mpr.uninterleave_data(PREFIX)

# resave data as zarr for dask processing
zarr_base_location = "../../data/inFrontEBrain"
zarr_paths = mpr.save_chan_data_to_zarr(PREFIX, zarr_base_location=zarr_base_location)

# open zarr files, adjust chunk size to be 10 MB - 1 GB based on sample rate/bit depth
raw = []
for zarr_path in zarr_paths:
    raw.append(xr.open_zarr(zarr_path, chunks="auto"))

### Enter processing parameters

In [ ]:
ZERO_SAMPLE_IDX = 0  # B205mini, fs = 56 MHz
# zero_sample_idx = 166 # B205mini, fs = 20 MHz

NSTACK = 1  # number of pulses to stack

MODIFY_RX_WINDOW = False  # set to true if you want to window the reference chirp only on receive, false uses ref chirp as transmitted in config file
RX_WINDOW = "rectangular"  # what you want to change the rx window to if modify_rx_window is true

# dielectric_constant = 3.17 # ice (air = 1, 66% velocity coax = 2.2957)
# dielectric_constant = 2.2957 # COAX (air = 1, 66% velocity coax = 2.2957)
DIELECTRIC_CONSTANT = 1  # air
SIG_SPEED = scipy.constants.c / np.sqrt(DIELECTRIC_CONSTANT)

### Generate reference chirp

In [ ]:
if MODIFY_RX_WINDOW:
    config = copy.deepcopy(raw[0].config)  # both channels have the same configs
    config["GENERATE"]["window"] = RX_WINDOW
else:
    config = raw[0].config

chirp_ts, ref_chirp = generate_chirp(config)

### View raw pulse in time domain to check for clipping

In [ ]:
single_pulse_raw = raw[0].radar_data[{"pulse_idx": 10}].compute()
plot1 = np.real(single_pulse_raw).hvplot.line(x="fast_time", color="red") * np.imag(
    single_pulse_raw
).hvplot.line(x="fast_time")

plot1 = plot1.opts(xlabel="Fast Time (s)", ylabel="Raw Amplitude")
plot1

### Clean and stack data

In [ ]:
stacked = pr.fill_errors(raw[0], error_fill_value=0.0)  # fill receiver errors with 0s

stacked = pr.stack(stacked, NSTACK).compute()  # stack, won't be used

### Apply FFT to data

In [ ]:
radar_fft_data = pr.radar_fft(
    stacked,
    ref_chirp,
    fs=stacked.config["GENERATE"]["sample_rate"],
    zero_sample_idx=ZERO_SAMPLE_IDX,
    signal_speed=SIG_SPEED,
)

fft_power = xr.apply_ufunc(
    lambda x: 20*np.log10(np.abs(x)),
    radar_fft_data,
    dask="parallelized"
).compute()

### View 1D FFT data

In [ ]:
plot1D = fft_power.radar_data[20,:].hvplot.line()
plot1D = plot1D * fft_power.radar_data[-1,:].hvplot.line()
# relevant options: xlim(-80,1000)

plot1D = plot1D.opts(xlabel='Reflection Distance (m)', ylabel='Return Power (dB)')
# plot1D.opts(xlim=(-50,200), ylim=(-120, -40), show_grid=True)
plot1D

### View 2D FFT data (radargram)

In [ ]:
# USING HOLOVIEWS (sometimes breaks)
print(fft_power.radar_data[20,20])
plot2D = fft_power.swap_dims({'pulse_idx': 'slow_time', 'travel_time': 'reflection_distance'}).hvplot.quadmesh(x='reflection_distance', y='slow_time', cmap='inferno', flip_yaxis=True)
# relevant options: ylim=(100,-50), clim=(-90,-40)

plot2D.opts(xlabel='Slow Time (s)', ylabel='Depth (m)', clabel='Return Power (dB)')
plot2D.opts(ylim=(-10, 70), clim=(-120, -40))

In [ ]:
# USING MATPLOTLIB (sometimes takes a while)
fig, ax = plt.subplots(1,1, figsize=(10,6), facecolor='white')

p = ax.pcolormesh( fft_power.reflection_distance, fft_power.slow_time, fft_power.radar_data, shading='nearest', cmap='inferno')
clb = fig.colorbar(p, ax=ax)
clb.set_label('Return Power (dB)')
ax.set_xlabel('Distance to Reflector (m)')
ax.set_ylabel('Slow Time (s)')
# relevant options: ax.set_ylim=(100,-50), ax.set_xlim=(0, 1), vmin=-90, vmax=40
# ax.set_ylim(100, -50)

### View spectrogram of stacked data

In [ ]:
inpt = raw[0]
inpt["radar_data"].shape

In [ ]:
num_presums = inpt.attrs["config"]["CHIRP"]["num_presums"]

In [ ]:
# data = stacked["radar_data"].to_numpy()
n = 1
normalize = True

pulse = pr.stack(inpt, n)[{"pulse_idx": 200}]["radar_data"].to_numpy()

f, t, S = scipy.signal.spectrogram(
    pulse,
    fs=inpt.attrs["config"]["GENERATE"]["sample_rate"],
    window="flattop",
    nperseg=128,
    noverlap=64,
    scaling="density",
    mode="psd",
    return_onesided=False,
)

if normalize:
    S /= np.max(S)

In [ ]:
fig, ax = plt.subplots(facecolor="white", figsize=(10, 6))
freq_mhz = (np.fft.fftshift(f) + inpt.attrs["config"]["RF0"]["freq"]) / 1e6
pcm = ax.pcolormesh(
    t, freq_mhz, 10 * np.log10(np.abs(np.fft.fftshift(S, axes=0))), shading="nearest"
)  #  vmin=-420, vmax=-200
clb = fig.colorbar(pcm, ax=ax)
clb.set_label("Power [dB]")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [MHz]")
# ax.set_title(f"Spectrogram of received data with n_stack={n}")
ax.text(
    0,
    1.05,
    PREFIX.split("/")[-1] + "\n" + f"n_stack * num_presums = {n * num_presums}",
    horizontalalignment="left",
    verticalalignment="center",
    transform=ax.transAxes,
    fontdict={"size": 12},
)
fig.tight_layout()
plt.show()

In [ ]:
# fig.savefig(f"orca_paper/outputs/{inpt.basename}_ft_spectrogram_n{n}.png", dpi=300)

### View Power Spectrum of All Received Data

In [ ]:
single_stack = pr.stack(inpt, inpt.radar_data.shape[1])

data_rx_fft = da.fft.fft(inpt.radar_data, axis=0) / inpt.radar_data.shape[0]
stacked_fft = da.fft.fft(stacked.radar_data, axis=0) / stacked.radar_data.shape[0]
full_fft = (
    da.fft.fft(single_stack.radar_data, axis=0) / single_stack.radar_data.shape[0]
)

data_rx_fft_pwr = 20 * da.log10(da.abs(data_rx_fft))
stacked_fft_pwr = 20 * da.log10(da.abs(stacked_fft))
full_fft_pwr = 20 * da.log10(da.abs(full_fft))

# data_rx_fft_pwr.shape

In [ ]:
# fig, axs = plt.subplots(2,1)
fig, axs = plt.subplots(facecolor="white", figsize=(10, 6))
freqs = np.fft.fftshift(
    np.fft.fftfreq(
        data_rx_fft_pwr.shape[0], d=1 / inpt.config["GENERATE"]["sample_rate"]
    )
)
axs.plot(freqs / 1e6, np.fft.fftshift(data_rx_fft_pwr[:, 0]), label="Single Pulse")
axs.plot(freqs / 1e6, np.fft.fftshift(stacked_fft_pwr[:, 0]), label="Single Stack")
axs.plot(freqs / 1e6, np.fft.fftshift(full_fft_pwr[:, 0]), label="Full File")
axs.set_xlabel("Frequency [MHz]")
axs.set_ylabel("Power [dB]")
axs.set_title("Spectrum -- Power")
axs.grid()
axs.legend()

# axs[1].plot(freqs/1e6, np.fft.fftshift(np.angle(data_rx_fft[:,0])))
# axs[1].plot(freqs/1e6, np.fft.fftshift(np.angle(stacked_fft[:,0])))
# axs[1].plot(freqs/1e6, np.fft.fftshift(np.angle(full_fft[:,0])))
# axs[1].set_xlabel('Frequency [MHz]')
# axs[1].set_ylabel('Phase [rad]')
# axs[1].set_title('Spectrum -- Phase')
# axs[1].grid()
# fig.tight_layout()